# One authorized last-bottleneck v3 stage
Append these TWO code cells to the **existing completed weighted-v2 GPU notebook**. A new runtime will not contain the required data/checkpoint/globals. Do not Run all or rerun historical training.
First cell: upload only vision_b1_lastblock_code_v3.zip; require actual 33/33 generated PASS. Save its evidence ZIP. Second cell: ONE authorized stage, from the pinned v2 best checkpoint. No new image upload or weight download. Stop on any error; no automatic retry.
Provisional labels; non-clinical; non-final; participant independence unverified. Original five-grade research scope and final Test remain unchanged.

In [ ]:
# NEW cell in the EXISTING completed weighted-v2 GPU runtime; no data upload or old-run replay.
from pathlib import Path
assert all(n in globals() for n in ('data_root','balanced_output','files')), 'STOP: retain the completed weighted-v2 runtime; do not rerun old training'
assert Path(data_root).resolve()==Path('/content/b1_pp1_pilot/development') and Path(data_root).is_dir()
assert Path(balanced_output).name=='pp1_b1_balanced_v2_20261015T150352287886Z'
import torch,hashlib,json,zipfile,io,tempfile,sys,os,stat,importlib
assert torch.cuda.is_available(), 'STOP: existing compatible GPU required; no installs or CPU fallback'
assert not Path('/content/drive').exists(), 'STOP: no Drive mount'
assert hashlib.sha256((Path(balanced_output)/'best.pt').read_bytes()).hexdigest()=='c84936ad7af7366e44e4ae2456c00ba748bae47b983ef9e0df204ce80c386dbf', 'STOP: exact prior best checkpoint required'
lastblock_upload=files.upload()  # ONLY vision_b1_lastblock_code_v3.zip; code/tests/aggregate references.
assert set(lastblock_upload)=={'vision_b1_lastblock_code_v3.zip'}
lastblock_raw=lastblock_upload['vision_b1_lastblock_code_v3.zip']
assert hashlib.sha256(lastblock_raw).hexdigest()=='c88c791d4e562012b44fad7095eaf814be7f442a303a53310da50a38caf26f2b'
lastblock_archive=zipfile.ZipFile(io.BytesIO(lastblock_raw))
assert lastblock_archive.testzip() is None
lastblock_manifest=json.loads(lastblock_archive.read('verification_manifest.json'))
assert len(lastblock_archive.namelist())==len(lastblock_manifest['files'])+1
assert set(lastblock_archive.namelist())==set(lastblock_manifest['files'])|{'verification_manifest.json'}
lastblock_root=Path(tempfile.mkdtemp(prefix='b1_lastblock_v3_',dir='/content'))
for info in lastblock_archive.infolist():
    name=info.filename; relative=Path(name)
    assert not relative.is_absolute() and '..' not in relative.parts and chr(92) not in name
    assert not info.is_dir() and not stat.S_ISLNK(info.external_attr>>16) and info.file_size<=1024*1024
    payload=lastblock_archive.read(name)
    if name in lastblock_manifest['files']: assert hashlib.sha256(payload).hexdigest()==lastblock_manifest['files'][name]
    target=lastblock_root/name; target.parent.mkdir(parents=True,exist_ok=True); target.write_bytes(payload)
sys.path.insert(0,str(lastblock_root));os.chdir(lastblock_root);importlib.invalidate_caches()
os.environ['PYTHONDONTWRITEBYTECODE']='1';sys.dont_write_bytecode=True
from src.vision import exploratory_pilot_b1_lastblock_v3 as lastblock
from src.vision import verify_b1_lastblock_runtime_v3 as lastblock_verify
for name,digest in lastblock.code_hashes().items(): assert digest==lastblock_manifest['files']['src/vision/'+name], 'STOP: cached/current code mismatch'
print('Fresh code-only extraction:',lastblock_root,'verified payloads:',len(lastblock_manifest['files']))
lastblock_summary=lastblock_verify.run(lastblock_root)
print(json.dumps(lastblock_summary,indent=2))
lastblock_evidence=lastblock_verify.evidence_zip(lastblock_root)
print('Synthetic evidence SHA256:',lastblock.legacy.sha256(lastblock_evidence))
files.download(str(lastblock_evidence))
assert lastblock_summary['status']=='PASS', 'STOP: all33 cases and all5 backend checks must pass, zero skips/errors/failures'
lastblock_proof=lastblock_verify.require_pass(lastblock_root/'lastblock_synthetic_evidence_v3')
print('33/33 generated checks passed. The next separate cell performs the ONE owner-authorized fine-tuning stage.')


In [ ]:
# Execute ONCE only after the preceding exact33-case generated PASS.
assert lastblock_verify.require_pass(lastblock_root/'lastblock_synthetic_evidence_v3')==lastblock_proof
owner_v3=json.loads((lastblock_root/lastblock_manifest['owner_record']).read_text())
assert lastblock.trainer.digest(lastblock.protocol())==owner_v3['protocol_sha256']
assert lastblock.trainer.digest(lastblock.code_hashes())==owner_v3['code_sha256']
from datetime import datetime,timezone
import gc,shutil
lastblock_run_id='pp1_b1_lastblock_v3_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
lastblock_output=Path('/content/b1_pp1_pilot/results')/lastblock_run_id
lastblock_auth={**owner_v3,'verification_sha256':lastblock_proof,'output_run_id':lastblock_run_id}
lastblock_auth.pop('runtime_bindings_pending',None)
gc.collect();torch.cuda.empty_cache()
print(lastblock.legacy.NOTICE,flush=True)
print('ONE stage from verified v2 best: final bottleneck convs+head, ALL BatchNorm frozen; max5 additional epochs, patience2.',flush=True)
lastblock_results=lastblock.run_pilot(data_root,Path(balanced_output)/'best.pt',lastblock_auth,lastblock_output,
                                     lastblock_root/'lastblock_synthetic_evidence_v3',opt_in=True)
assert lastblock_verify.snapshot(lastblock_root,lastblock_manifest['files'])==lastblock_manifest['files']
unweighted_ref=json.loads((lastblock_root/lastblock_manifest['references'][0]).read_text())
weighted_ref=json.loads((lastblock_root/lastblock_manifest['references'][1]).read_text())
lastblock_comparison={'notice':lastblock.legacy.NOTICE,'unweighted_historical_validation':unweighted_ref['validation'],
    'weighted_historical_validation':weighted_ref['independent_probability_metric_calculations'],
    'new_stage_best_epoch':lastblock_results['best_epoch'],'new_stage_validation':lastblock_results['best_validation'],
    'optimizer_groups':lastblock_results['optimizer_groups'],'frozen_tensor_checks':lastblock_results['frozen_tensor_checks'],
    'measured_runtime':lastblock_results['measured_runtime'],
    'selection_optimism':'Repeated Validation/checkpoint selection; not final held-out or clinical evaluation',
    'participant_independence_verified':False,'qualified_labels_approved':False,'test_used':False}
lastblock.base.bundle.write_json(lastblock_output/'comparison.json',lastblock_comparison)
shutil.copyfile(lastblock_root/lastblock_manifest['owner_record'],lastblock_output/'owner_record.json')
shutil.copytree(lastblock_root/'lastblock_synthetic_evidence_v3',lastblock_output/'synthetic_verification')
for name in ('source_hashes_before.json','source_hashes_after.json'):
    lastblock.base.bundle.write_json(lastblock_output/name,lastblock_manifest['files'])
assert len(list((lastblock_output/'gradcam').glob('*/*.png')))==9
assert len(json.loads((lastblock_output/'validation_predictions_best.json').read_text())['records'])==53
for name in ('best.pt','last.pt','initial_tensor_hashes.json','selected_tensor_hashes.json','pilot_results.json'):
    assert (lastblock_output/name).is_file()
lastblock_hashes={str(p.relative_to(lastblock_output)):lastblock.legacy.sha256(p) for p in sorted(lastblock_output.rglob('*')) if p.is_file()}
lastblock.base.bundle.write_json(lastblock_output/'results_file_hashes.json',lastblock_hashes)
print(json.dumps(lastblock_comparison,indent=2,allow_nan=False))
lastblock_results_zip=shutil.make_archive('/content/'+lastblock_run_id+'_results','zip',lastblock_output)
print('Actual new-stage results ZIP:',lastblock_results_zip,'SHA256:',lastblock.legacy.sha256(lastblock_results_zip))
files.download(lastblock_results_zip)
